# 모듈 2. AI 확장하기 — Function & Tool Calling

> LLM이 외부 함수를 "호출 결정"하고, 개발자가 "실제 실행"하는 Tool Calling의 전체 흐름을 실습합니다.

**학습 목표**
- OpenAI `tools` 파라미터를 이용한 함수 스펙 정의
- 단일/다중 도구 등록 및 모델의 자동 호출 판단
- tool_calls 응답 → 실제 함수 실행 → 2차 호출의 전체 워크플로우
- LangChain bind_tools()를 이용한 통합 도구 바인딩

## 1.OpenAI SDK 기반 function calling

### 1)단일 도구 정의 — 날씨 조회

- 1차 호출 → 함수 호출 요청 확인 → 실제 함수 실행 → 함수 결과를 2차 호출로 전달 → 최종 답변 생성

In [2]:
# 1. 필수 모듈 임포트
import os
import json
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv

# 2. 환경변수 로드 & 클라이언트 생성
load_dotenv(find_dotenv(), override=True)
client = OpenAI()

# 3. 베이스 모델 및 사용자 입력 값 지정
MODEL = "gpt-4o-mini"

city = "서울"

# 4. 일반 대환 호출: function/tools 없이 순수 질문
resp_plain = client.responses.create(
    model=MODEL,
    input=f"{city}의 날씨는 어때",
    temperature=0
)
print("일반 응답:", resp_plain.output_text)

일반 응답: 현재 서울의 날씨에 대한 실시간 정보는 제공할 수 없지만, 일반적으로 서울의 날씨는 계절에 따라 다릅니다. 봄과 가을은 온화하고 쾌적하며, 여름은 덥고 습하고, 겨울은 춥고 건조합니다. 특정 날짜의 날씨를 알고 싶다면 기상청 웹사이트나 날씨 앱을 확인해 보세요!


- https://developers.openai.com/api/docs/guides/function-calling

In [ ]:
# 5. 호출할 실제 함수 정의
def get_weather(location: str) -> str:
    """오늘의 날씨를 반환하는 모의 함수. 실서비스에서는 실제 날씨 API와 연동한다."""
    weather_data = {
        "서울": "맑고 기온은 24도입니다.",
        "Seoul": "맑고 기온은 24도입니다.",
        "New York": "비가 오고 있으며 기온은 15도입니다.",
        "Tokyo": "흐림, 기온은 20도입니다."
    }

    return weather_data.get(
        location,
        f"{location}의 날씨 정보를 찾을 수 없습니다."
    )

# 6. 모델에 노출할 툴 스펙 정의 (JSON Schema)
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "오늘의 날씨를 알려주는 함수",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "날씨를 알고 싶은 지역명. 예: 서울, Seoul",
                }
            },
            "required": ["location"],
            "additionalProperties": False,
        },
    }
]

# 7. 1차 호출: 모델이 툴 호출 여부를 판단
# 모델은 실제 함수를 실행하지 않고, 호출할 함수명과 인자만 생성한다.
response = client.responses.create(
    model = MODEL,
    input = f"{city}의 날씨는 어때?",
    tools = tools,
    tool_choice = "auto",
    temperature = 0
)

# 8. 1차 응답 확인
print("\n[1차 응답 텍스트]")
print(response.output_text)

print("\n[1차 응답 구조]")
for item in response.output:
    print("type:", item.type)
    print("name:", getattr(item, "name", None))
    print("arguments:", getattr(item, "arguments", None))
    print()


[1차 응답 텍스트]


[1차 응답 구조]
type: function_call
name: get_weather
arguments: {"location":"서울"}



In [5]:
# 9. function_call 찾기
tool_call = next(
    (item for item in response.output if item.type == "function_call"),
    None
)

# 10. 함수 호출이 필요한 경우 처리
if tool_call:
    print("\n[함수 호출 감지]")
    print("함수 이름:", tool_call.name)
    print("전달된 인자(JSON):", tool_call.arguments)

    # 등록된 함수 목록
    available_functions = {
        "get_weather": get_weather,
    }

    # 알 수 없는 함수명 방지
    if tool_call.name not in available_functions:
        raise ValueError(f"지원하지 않는 함수입니다: {tool_call.name}")

    # JSON 문자열을 Python dict로 변환
    try:
        args = json.loads(tool_call.arguments)
    except json.JSONDecodeError as e:
        raise ValueError(f"함수 인자 JSON 파싱에 실패했습니다: {tool_call.arguments}") from e

    print("변환된 인자(dict):", args)

    # 실제 Python 함수 실행
    function_to_call = available_functions[tool_call.name]
    result = function_to_call(**args)

    print("\n[함수 실행 결과]")
    print(result)

    # 11. 함수 실행 결과를 모델에 다시 전달하여 최종 응답 생성
    final_response = client.responses.create(
        model=MODEL,
        previous_response_id=response.id,
        input=[
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result
            }
        ],
        temperature=0
    )

    print("\n[최종 응답]")
    print(final_response.output_text)

# 12. 함수 호출 없이 바로 답변한 경우
else:
    print("\n[함수 호출 없이 바로 응답]")
    print(response.output_text)



[함수 호출 감지]
함수 이름: get_weather
전달된 인자(JSON): {"location":"서울"}
변환된 인자(dict): {'location': '서울'}

[함수 실행 결과]
맑고 기온은 24도입니다.

[최종 응답]
현재 서울의 날씨는 맑고 기온은 24도입니다. 외출하기 좋은 날씨네요!


## 2.LangChain 기반 Toll Calling

> **[LangChain 1.0 변경사항]**  
> - LangChain에서도 `bind_tools()` 메서드를 사용하여 도구를 바인딩  
> - Pydantic 모델, dict, LangChain Tool 등 다양한 형태의 도구를 통일된 방식으로 전달 가능  
> - `from langchain_core.tools import tool` 데코레이터로 간단하게 도구 정의 가능

- 전체 흐름 설명 ---------------------------
 1. 사용잘 질문 입력
 2. LLM이 tool 필요 여부 판단
 3. tool_calls 생성: @tool 데코레이터를 사용하면 Python 함수가 LLM에서 호출 가능한 tool로 변환됨
 4. 개발자가 함수 실행
 5. 결과를 활용

In [2]:
# 1. 라이브러리 import
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

# 2. tool 함수 정의
@tool
def get_weather(location: str) -> str:
    """특정 지역의 날씨를 알려주는 함수"""
    weather_data = {
        "서울": "맑고 24도 입니다.",
        "뉴욕": "비가 오고 있으며, 15도 입니다."
    }
    return weather_data.get(location, "해당 지역의 날씨 정보가 없습니다.")

# 3. LLM 모델 생성
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

# 4. tool을 LLM에 연결
llm_with_tools = llm.bind_tools([get_weather])

# 5. 사용자 입력 정의
messages = [HumanMessage(content='서울 날씨 어때?')]

# 6. 모델 호출
response = llm_with_tools.invoke(messages)

# 7. 모델 응답 확인
print("텍스트 응답:", response.content)
print("tool 호출 정보:", response.tool_calls)

# 8. tool 호출 실행
if response.tool_calls:
    tool_call = response.tool_calls[0]

    print("\n[tool 호출 감지]")
    print("호출된 함수:", tool_call["name"])
    print("전달된 인자:", tool_call["args"])

    result = get_weather.invoke(tool_call["args"])

    print("\n[tool 실행 결과]")
    print(result)
else:
    print("\n[tool 호출 없음]")
    print(response.content)


텍스트 응답: 
tool 호출 정보: [{'name': 'get_weather', 'args': {'location': '서울'}, 'id': 'call_QLVn1EUoGuyHWYG5pvdc4C1V', 'type': 'tool_call'}]

[tool 호출 감지]
호출된 함수: get_weather
전달된 인자: {'location': '서울'}

[tool 실행 결과]
맑고 24도 입니다.


## OpenAI SDK Tool vs LangChain

- OpenAI SDK에서는 tools 스키마를 직접 정의하고 tool_call 결과를 파싱한다.
- LangChain에서는 @tool 데코레이터로 Python 함수를 tool 객체로 변환하고, bind_tools()로 LLM에 연결한다. 모델이 tool 사용이 필요하다고 판단하면 response.tool_calls에 호출할 함수명과 인자를 담아 반환한다.

### 미션 : 실시간 날씨 정보 검색해서 적용하기
- https://openweathermap.org/ API_KEY 얻기
- 도시이름 입력 받아 처리하기

In [4]:
# 1. 라이브러리 선언
import os
import requests # OpenWeatherMap 서버에 실제 요청을 보내기 위해 필요함
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# 2. .env 파일에서 API 키와 모델 불러오기
load_dotenv(find_dotenv(), override = True)
OPENWEATHER_API_KEY =  os.getenv("OPENWEATHER_API_KEY")

# 3. tool 함수 정의하기
@tool
def get_weather(location: str) -> str:
    """api key를 통해 특정 지역의 날씨를 정보를 불러오는 함수"""
    url = "https://api.openweathermap.org/data/2.5/weather"
    params = {"q": location, "appid": OPENWEATHER_API_KEY, "units": "metric", "lang": "kr"}

    resp = requests.get(url, params=params)

    if resp.status_code != 200:
        return f"{location}의 날씨 정보를 가져오지 못했습니다."

    data = resp.json()
    desc = data["weather"][0]["description"]
    temp = data["main"]["temp"]

    return f"{location}: {desc}, 현재 기온 {temp}도"

# 4. LLM 모델 생성
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.6)

# 5. LLM에 도구 연결
llm_with_tools = llm.bind_tools([get_weather])

# 6. 도시 입력받기
city = input("날씨를 확인할 도시를 입력하세요: ")

# 7. 1차 호출: LLM이 도구 필요 여부 판단
messages = [HumanMessage(content=f"{city}의 날씨는 어때?")]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# 8. 도구 실행 + 결과를 대화 기록에 추가
for tool_call in ai_msg.tool_calls:
    print("\n[함수 호출 감지]", tool_call["name"], tool_call["args"])
    tool_output = get_weather.invoke(tool_call["args"])   # 실제 실행
    print("[함수 실행 결과]", tool_output)
    messages.append(ToolMessage(content=tool_output, tool_call_id = tool_call["id"]))

# 7. 2차 호출: 결과를 자연어로 정리
final = llm_with_tools.invoke(messages)
print("\n[최종 응답]")
print(final.content)


[함수 호출 감지] get_weather {'location': 'Seoul'}
[함수 실행 결과] Seoul: 맑음, 현재 기온 28.19도

[최종 응답]
서울의 날씨는 맑고, 현재 기온은 28.19도입니다.
